# CANDLE V2 — fresh training with peer prototypes

Enable **Internet** and a **GPU** in Kaggle. One GPU is used by this runner.
Fresh seed42, Tasks0–5, 20 rounds/task, xi=0.8, batch2048 by default.
One personalized model/client; decentralized neighborhood aggregation.

Changes from V1: peer-prototype representation alignment, virtual missing-class
CE and current-input teacher preservation. The registered-router repair uses the
saved binary_cosine detector to select task from input, then masks cosine FC2
to saved available classes in class_tasks for that task. Context-specific
adapters are disabled. No true task labels are provided at inference.
No raw historical replay, CGoFed, CME, APPLIANCE, or donor models at inference.

Uses the same content-disjoint V1 BASE training split and internal local holdout;
**not 100% original TRAIN**. No CAL/FIT/TEST drives prototype fitting.
Round r post-aggregation prototypes train round r+1; old-task banks are discarded.

Checkpoint each round, ZIP/task, 12GiB output guard. Source/roles live in
`/kaggle/temp`; only results are exported from `/kaggle/working`.
Only terminal Task5/Round19 is evaluated, on the full34-class TEST partitioned
across receiver models exactly once, matching V1 sample allocation.
Read `candle_v2_completion.json` for Accuracy and Macro-F1.

This is an updated inference policy; the training objective is unchanged.
Development checkpoint diagnostics showed improvement with remaining breaks
and lower new-class recall. Full TEST gains and historical safety are unproven.
For existing results (14), use eval_candle_v2_repair_kaggle.ipynb without retraining.
The notebook clones pinned GitHub source; no embedded source/runtime ZIP.


In [ ]:
# CANDLE V2: training-time prototype alignment and injection.
import os
from pathlib import Path
import subprocess
import sys

CODE_REF = "97e922ccab67f433ba07793cf0f2e23738bab144"
DATA_DIR = "/kaggle/input/datasets/khoilv2005/100-clients"
OUTPUT_DIR = "/kaggle/working/results_candle_v2_xi_0p8_seed_42"
# Reuse a mounted V1 roles directory if available; otherwise prepare the same split.
ROLES_DIR = "/kaggle/temp/candle_v2_roles"
BATCH_SIZE = 2048
INFERENCE_POLICY = "binary_cosine_registered_v1"
# Empty => fresh Task0..5. For resume, mount a CANDLE V2 continuation/task ZIP.
RESUME_STATE = ""

env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1", "PYTHONUNBUFFERED": "1"}
code = Path("/kaggle/temp/candle_v2_source")
code.parent.mkdir(parents=True, exist_ok=True)
if not code.exists():
    subprocess.check_call(["git", "clone", "--filter=blob:none", "--no-checkout",
        "https://github.com/khoilv2005/FL_IL_IDS.git", str(code)], env=env)
    subprocess.check_call(["git", "sparse-checkout", "set", "--no-cone",
        "/fed_learning/", "/appliance/", "/tools/", "/configs/",
        "/train_incremental_kaggle.py", "/eval_checkpoint.py",
        "/requirements-candle-v1-kaggle.txt"], cwd=code, env=env)
    subprocess.check_call(["git", "checkout", "--detach", CODE_REF], cwd=code, env=env)
actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=code, text=True).strip()
if actual != CODE_REF:
    raise ValueError("CANDLE V2 source differs from pinned revision; use a fresh session")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r",
    str(code / "requirements-candle-v1-kaggle.txt")])
print("CANDLE V2 source:", actual, flush=True)
command = [sys.executable, "-u", str(code / "tools/train_candle_v2.py"),
    "--data-dir", DATA_DIR, "--output", OUTPUT_DIR, "--roles", ROLES_DIR,
    "--batch-size", str(BATCH_SIZE), "--inference-policy", INFERENCE_POLICY]
if RESUME_STATE:
    command += ["--resume", RESUME_STATE]
subprocess.check_call(command, cwd=code, env=env)
print(Path(OUTPUT_DIR, "candle_v2_completion.json").read_text(encoding="utf-8"))
